# VADER Sentiment Analysis Pipeline

**NLP/ML Engineer: Kevin**  
**Course: CAP 3764 Advanced Data Science - FIU**

This notebook performs sentiment analysis on news articles using VADER (Valence Aware Dictionary and sEntiment Reasoner).

## Setup and Imports

In [1]:
import os
import random
from typing import List, Dict

import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADER lexicon if not already downloaded
try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

## Define Helper Functions

In [2]:
def initialize_vader() -> SentimentIntensityAnalyzer:
    analyzer = SentimentIntensityAnalyzer()

    test_texts = [
        "This is excellent news! Stock prices soared.",
        "The company reported terrible losses and declining revenue.",
        "The quarterly report was released today."
    ]
    
    print("\n" + "="*50)
    print("VADER INITIALIZATION TEST")
    print("="*50)
    
    for text in test_texts:
        scores = analyzer.polarity_scores(text)
        print(f"\nText: {text}")
        print(f"  Compound: {scores['compound']:+.3f} | "
              f"Pos: {scores['pos']:.3f} | "
              f"Neu: {scores['neu']:.3f} | "
              f"Neg: {scores['neg']:.3f}")
    
    print("\nVADER is working!\n")
    return analyzer

In [3]:
def categorize_sentiment(compound_score: float) -> str:
    if compound_score >= 0.05:
        return 'positive'
    elif compound_score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

In [4]:
def score_articles(articles_df: pd.DataFrame, analyzer: SentimentIntensityAnalyzer) -> pd.DataFrame:

    print("="*50)
    print(f"SCORING {len(articles_df)} ARTICLES")
    print("="*50)
    
    results = []
    
    for idx, row in articles_df.iterrows():
        headline = str(row.get('headline', '')).strip()
        content = str(row.get('content', '')).strip()
        
        headline_scores = analyzer.polarity_scores(headline) if headline else {'compound': 0.0}
        content_scores = analyzer.polarity_scores(content) if content else {'compound': 0.0}
        
        combined_compound = (0.6 * headline_scores['compound'] + 
                            0.4 * content_scores['compound'])
        
        results.append({
            'ticker': row.get('ticker', ''),
            'date': row.get('date', ''),
            'headline': headline,
            'content': content,
            'source': row.get('source', ''),
            'headline_sentiment': headline_scores['compound'],
            'content_sentiment': content_scores['compound'],
            'sentiment_compound': combined_compound,
            'sentiment_label': categorize_sentiment(combined_compound)
        })
        
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(articles_df)} articles...")
    
    print(f"Completed scoring all {len(articles_df)} articles\n")
    
    return pd.DataFrame(results)

In [5]:
def validate_sentiment_sample(scored_df: pd.DataFrame, n_samples: int = 20) -> List[Dict]:
    print("="*50)
    print(f"VALIDATION SAMPLE ({n_samples} ARTICLES)")
    print("="*50)
    
    # Sample diverse sentiment scores
    positive = scored_df[scored_df['sentiment_label'] == 'positive'].sample(
        min(7, len(scored_df[scored_df['sentiment_label'] == 'positive'])), 
        random_state=42
    )
    negative = scored_df[scored_df['sentiment_label'] == 'negative'].sample(
        min(7, len(scored_df[scored_df['sentiment_label'] == 'negative'])), 
        random_state=42
    )
    neutral = scored_df[scored_df['sentiment_label'] == 'neutral'].sample(
        min(6, len(scored_df[scored_df['sentiment_label'] == 'neutral'])), 
        random_state=42
    )
    
    validation_sample = pd.concat([positive, negative, neutral]).reset_index(drop=True)
    
    validation_results = []
    
    for idx, row in validation_sample.iterrows():
        print(f"\n{'─'*50}")
        print(f"Sample {idx + 1}/{len(validation_sample)}")
        print(f"{'─'*50}")
        print(f"Ticker: {row['ticker']}")
        print(f"Date: {row['date']}")
        print(f"Source: {row['source']}")
        print(f"\nHeadline: {row['headline'][:150]}")
        if len(row['headline']) > 150:
            print("...")
        print(f"\nContent Preview: {row['content'][:200]}")
        if len(row['content']) > 200:
            print("...")
        print(f"\n VADER Scores:")
        print(f"   Headline Sentiment: {row['headline_sentiment']:+.3f}")
        print(f"   Content Sentiment:  {row['content_sentiment']:+.3f}")
        print(f"   Combined Score:     {row['sentiment_compound']:+.3f}")
        print(f"   Label: {row['sentiment_label'].upper()}")
        
        validation_results.append({
            'sample_id': idx + 1,
            'ticker': row['ticker'],
            'date': row['date'],
            'headline': row['headline'],
            'sentiment_compound': row['sentiment_compound'],
            'sentiment_label': row['sentiment_label']
        })
    
    return validation_results

In [6]:
def generate_summary_stats(scored_df: pd.DataFrame) -> pd.DataFrame:
    print("\n" + "="*50)
    print("SENTIMENT SUMMARY STATISTICS")
    print("="*50)
    
    # Overall stats
    print("\n Overall Distribution:")
    label_counts = scored_df['sentiment_label'].value_counts()
    for label, count in label_counts.items():
        pct = (count / len(scored_df)) * 100
        print(f"   {label.capitalize():8s}: {count:3d} ({pct:5.1f}%)")
    
    print(f"\n Overall Sentiment Metrics:")
    print(f"   Mean compound score: {scored_df['sentiment_compound'].mean():+.4f}")
    print(f"   Median compound score: {scored_df['sentiment_compound'].median():+.4f}")
    print(f"   Std deviation: {scored_df['sentiment_compound'].std():.4f}")
    
    # By ticker
    print("\n By Ticker:")
    ticker_stats = scored_df.groupby('ticker').agg({
        'sentiment_compound': ['mean', 'median', 'std', 'count'],
        'sentiment_label': lambda x: (x == 'positive').sum()
    }).round(4)
    
    for ticker in scored_df['ticker'].unique():
        ticker_data = scored_df[scored_df['ticker'] == ticker]
        print(f"\n   {ticker}:")
        print(f"      Articles: {len(ticker_data)}")
        print(f"      Mean sentiment: {ticker_data['sentiment_compound'].mean():+.4f}")
        print(f"      Positive: {(ticker_data['sentiment_label'] == 'positive').sum()} "
              f"({(ticker_data['sentiment_label'] == 'positive').sum() / len(ticker_data) * 100:.1f}%)")
        print(f"      Neutral:  {(ticker_data['sentiment_label'] == 'neutral').sum()} "
              f"({(ticker_data['sentiment_label'] == 'neutral').sum() / len(ticker_data) * 100:.1f}%)")
        print(f"      Negative: {(ticker_data['sentiment_label'] == 'negative').sum()} "
              f"({(ticker_data['sentiment_label'] == 'negative').sum() / len(ticker_data) * 100:.1f}%)")
    
    return ticker_stats

## Main Pipeline Execution

In [7]:
# Initialize VADER analyzer
print("\n" + "="*50)
print("VADER SENTIMENT ANALYSIS PIPELINE")
print("="*50 + "\n")

analyzer = initialize_vader()


VADER SENTIMENT ANALYSIS PIPELINE


VADER INITIALIZATION TEST

Text: This is excellent news! Stock prices soared.
  Compound: +0.611 | Pos: 0.400 | Neu: 0.600 | Neg: 0.000

Text: The company reported terrible losses and declining revenue.
  Compound: -0.700 | Pos: 0.000 | Neu: 0.508 | Neg: 0.492

Text: The quarterly report was released today.
  Compound: +0.000 | Pos: 0.000 | Neu: 1.000 | Neg: 0.000

VADER is working!



In [10]:
# Load articles data
articles_path = '../data/raw/articles.csv'

if not os.path.exists(articles_path):
    raise FileNotFoundError(f"articles.csv not found at {articles_path}")
    
articles = pd.read_csv(articles_path)
print(f"Loaded {len(articles)} articles from {articles_path}")

# Remove duplicates
original_count = len(articles)
articles = articles.drop_duplicates(subset=['ticker', 'date', 'headline'], keep='first')
articles = articles.dropna(subset=['headline', 'content'])
articles = articles[articles['headline'].str.strip().str.len() > 0]
articles = articles.reset_index(drop=True)

duplicates_removed = original_count - len(articles)
print(f"Removed {duplicates_removed} duplicates/invalid rows")
print(f"Processing {len(articles)} unique articles")

Loaded 1167 articles from ../data/raw/articles.csv
Removed 0 duplicates/invalid rows
Processing 1167 unique articles


In [11]:
# Score all articles with sentiment
scored_articles = score_articles(articles, analyzer)

SCORING 1167 ARTICLES
Processed 50/1167 articles...
Processed 100/1167 articles...
Processed 150/1167 articles...
Processed 200/1167 articles...
Processed 250/1167 articles...
Processed 300/1167 articles...
Processed 350/1167 articles...
Processed 400/1167 articles...
Processed 450/1167 articles...
Processed 500/1167 articles...
Processed 550/1167 articles...
Processed 600/1167 articles...
Processed 650/1167 articles...
Processed 700/1167 articles...
Processed 750/1167 articles...
Processed 800/1167 articles...
Processed 850/1167 articles...
Processed 900/1167 articles...
Processed 950/1167 articles...
Processed 1000/1167 articles...
Processed 1050/1167 articles...
Processed 1100/1167 articles...
Processed 1150/1167 articles...
Completed scoring all 1167 articles



In [12]:
# Generate summary statistics
summary_stats = generate_summary_stats(scored_articles)


SENTIMENT SUMMARY STATISTICS

 Overall Distribution:
   Positive: 956 ( 81.9%)
   Negative: 135 ( 11.6%)
   Neutral :  76 (  6.5%)

 Overall Sentiment Metrics:
   Mean compound score: +0.2869
   Median compound score: +0.3141
   Std deviation: 0.3144

 By Ticker:

   AAPL:
      Articles: 1167
      Mean sentiment: +0.2869
      Positive: 956 (81.9%)
      Neutral:  76 (6.5%)
      Negative: 135 (11.6%)


In [13]:
# Validate with sample articles
validation_samples = validate_sentiment_sample(scored_articles, n_samples=20)

VALIDATION SAMPLE (20 ARTICLES)

──────────────────────────────────────────────────
Sample 1/20
──────────────────────────────────────────────────
Ticker: AAPL
Date: 2023-11-05
Source: The Korea Economic Daily Global Edition

Headline: On-device AI chips: New battleground for chipmakers

Content Preview: Global chipmakers like Samsung Electronics, Qualcomm, Intel, and AMD are intensifying their focus on on-device AI chips for smartphones, laptops, and PCs, recognizing its potential for customized serv
...

 VADER Scores:
   Headline Sentiment: -0.402
   Content Sentiment:  +0.836
   Combined Score:     +0.093
   Label: POSITIVE

──────────────────────────────────────────────────
Sample 2/20
──────────────────────────────────────────────────
Ticker: AAPL
Date: 2024-11-14
Source: Bloomberg.com

Headline: Buffett’s Berkshire Buys Stakes in Domino’s Pizza, Pool Corp. (DPZ, BRK/A): 13F

Content Preview: Berkshire Hathaway Inc. acquired stakes in Domino’s Pizza Inc. and Pool Corp. during the

In [ ]:
# Save output files
# Rename sentiment_compound to vader_compound for consistency with analysis notebook
output_df = scored_articles.copy()
output_df = output_df.rename(columns={'sentiment_compound': 'vader_compound'})

# Save to data folder with the expected name
output_path = '../data/sentiment_articles.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\n Saved sentiment_articles.csv to {output_path}")

# Also save validation sample
validation_df = pd.DataFrame(validation_samples)
validation_path = '../data/validation_sample.csv'
validation_df.to_csv(validation_path, index=False, encoding='utf-8')
print(f"Saved validation_sample.csv to {validation_path}")

print("\n" + "="*50)
print("PIPELINE COMPLETE!")
print("="*50)
print("\nOutput files:")
print(f"  • {output_path}: All articles with sentiment scores")
print(f"  • {validation_path}: 20 sample articles for manual review")
print("\n")